# Open Pit Mining

### GENERAL

In [1]:
import ee
import geemap
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

ee.Authenticate()
ee.Initialize(project="mining-457808")


Successfully saved authorization token.


In [2]:
with open("niedzwiedzia_gora_polygon.geojson") as f:
    geojson_data = json.load(f)

with open("niedzwiedzia_gora_buildings.geojson") as f:
    buildings_geojson = json.load(f)
    
polygon = geemap.geojson_to_ee(geojson_data)
buildings = geemap.geojson_to_ee(buildings_geojson)

In [24]:
#na zas

# rectangle_case = polygon.geometry().buffer(500).bounds()
# bounds = rectangle_case.bounds().getInfo()['coordinates'][0]

# lats = [pt[1] for pt in bounds]
# lons = [pt[0] for pt in bounds]

# south = min(lats)
# north = max(lats)
# west = min(lons)
# east = max(lons)

# print(f"{south}, {west}, {north}, {east}")

In [3]:
# TODO ogarnąć daty bez sniegu

date_list_start = pd.date_range(start='2016-01-01', end='2025-03-01', freq='MS')
date_list_start = [date.strftime('%Y-%m-%d') for date in date_list_start]
date_list_end = pd.date_range(start='2016-01-31', end='2025-03-31', freq='M')
date_list_end = [date.strftime('%Y-%m-%d') for date in date_list_end]

selected_months = ['05', '06','07', '08', '09', '10']

filtered_date_list_start = [d for d in date_list_start if d[5:7] in selected_months]
filtered_date_list_end = [d for d in date_list_end if d[5:7] in selected_months]


C:\Users\Hp\AppData\Local\Temp\ipykernel_19484\3375753686.py:5: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  date_list_end = pd.date_range(start='2016-01-31', end='2025-03-31', freq='M')


### ONE YEAR

In [4]:
image_case = (
    ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
    .filterBounds(polygon.geometry())
    .filterDate(f'2024-07-01', f'2024-07-31')
    .sort('CLOUDY_PIXEL_PERCENTAGE')
    .first()
)

In [86]:
# Przygotowanie obszaru
rectangle_case = polygon.geometry().buffer(500).bounds()
image_case = image_case.clip(rectangle_case)

# NDVI i bare soil
ndvi_case = image_case.normalizedDifference(['B8', 'B4'])
bare_soil_case = ndvi_case.lt(0.18)

# Wyznaczenie exceeded poza polygonem
ones_case = ee.Image.constant(1).clip(rectangle_case)
polygon_mask_case = ee.Image.constant(1).clip(polygon)
outside_mask_case = ones_case.where(polygon_mask_case.eq(1), 0)
exceeded_case = bare_soil_case.updateMask(outside_mask_case)

# -----------------------------
# Jedna warstwa: bare soil = 1, exceeded = 2
# -----------------------------
bare_soil_val = bare_soil_case.multiply(1)
exceeded_val = exceeded_case.multiply(2)
combined_mask = bare_soil_val.where(exceeded_val.eq(2), 2)

# -----------------------------
# Wizualizacja
# -----------------------------
combined_vis = {
    'min': 0,
    'max': 2,
    'palette': ['00000000', 'FFFFFF', 'FFFF00']  # 00000000 = przezroczyste, biały = bare soil, żółty = exceeded
}

# -----------------------------
# Map
# -----------------------------
Map = geemap.Map()
Map.centerObject(polygon, zoom=15)
Map.add_layer(combined_mask, combined_vis, "Bare soil + Exceeded")
Map


Map(center=[50.10339412565508, 19.63649860680545], controls=(WidgetControl(options=['position', 'transparent_b…

In [5]:
rectangle_case = polygon.geometry().buffer(500).bounds()
image_case = image_case.clip(rectangle_case)

rgb_case = image_case.select(['B4', 'B3', 'B2'])
ndvi_case = image_case.normalizedDifference(['B8', 'B4'])
bare_soil_case = ndvi_case.lt(0.18)

ones_case = ee.Image.constant(1).clip(rectangle_case)
polygon_mask_case = ee.Image.constant(1).clip(polygon)
outside_mask_case = ones_case.where(polygon_mask_case.eq(1), 0)
exceeded_case = bare_soil_case.updateMask(outside_mask_case)

buildings_mask_case = ee.Image().byte().paint(featureCollection=buildings, color=1).clip(rectangle_case)
buildings_mask_case = ones_case.where(buildings_mask_case.eq(1), 0)

exceeded_case = exceeded_case.updateMask(buildings_mask_case)

In [60]:
polygon_style = {"color": "red", "fillColor": "00000000", "width": 1}
buildings_style = {"color": "green", "fillColor": "00000000", "width": 1}
ndvi_vis = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
rgb_vis = {'min': 0, 'max': 3000}
bare_soil_vis = {'min': 0, 'max': 1, 'palette': ['black', 'white']}
exceeded_vis = {'min': 0, 'max': 1, 'palette': ['black', 'yellow']}

Map = geemap.Map()
Map.centerObject(polygon, zoom=15)
Map.setOptions('SATELLITE')
Map.add_layer(rgb_case, rgb_vis, "orthophotomap")
Map.add_layer(ndvi_case, ndvi_vis, "NDVI")
Map.add_layer(bare_soil_case, bare_soil_vis, "bare soil mask")
Map.add_layer(exceeded_case, exceeded_vis, "exceeded")
Map.add_layer(polygon.style(**polygon_style), {}, "mining area")
Map.add_layer(buildings.style(**buildings_style), {}, "buildings")
Map

Map(center=[50.10339412565508, 19.63649860680545], controls=(WidgetControl(options=['position', 'transparent_b…

## SAR

In [ ]:
image_start_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(polygon)
    .filterDate('2023-06-01', '2023-06-11')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .select('VV')
)

image_end_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(polygon)
    .filterDate('2023-06-19', '2023-06-30')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .select('VV')
)


In [50]:
image_start = image_start_collection.median()

last_date_start = ee.Date(image_start_collection.aggregate_max('system:time_start'))
start_date = last_date_start.format('dd-MM-YYYY').getInfo()
print("Ostatnia data pierwszej mediany:", start_date)

image_end = image_end_collection.median()

first_date_end = ee.Date(image_end_collection.aggregate_min('system:time_start'))
end_date = first_date_end.format('dd-MM-YYYY').getInfo()
print("Pierwsza data drugiej mediany:", end_date)

Ostatnia data pierwszej mediany: 08-06-2023
Pierwsza data drugiej mediany: 27-06-2023


In [54]:
image_start_case = image_start.clip(polygon)
image_end_case = image_end.clip(polygon)

In [78]:
log_ratio = image_end_case.divide(image_start_case).log()
log_ratio = log_ratio.clip(polygon)

kernel = ee.Kernel.square(radius=1) 
log_ratio_smooth = log_ratio.focal_mean(kernel=kernel, iterations=1)

change_class = log_ratio_smooth.expression(
    "(b > 0.4) ? 1"      # nasyp
    ": (b < -0.4) ? -1" # kopanie
    ": 0",
    {"b": log_ratio_smooth}
)
change_class = change_class.clip(polygon)

In [79]:
sar_style = {'min': -25, 'max': 0}
log_vis = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'red']}
buildings_style = {"color": "black", "fillColor": "aaaaaa", "width": 1}

title = widgets.HTML(
    value=f"<h3 style='text-align:center'>Changes in landscape from {start_date} to {end_date} in Niedźwiedzia Góra mine</h3>"
)

legend_dict = {
    'Digging (-1)': 'blue',
    'No change (0)': 'white',
    'Mound (+1)': 'red'
}

Map = geemap.Map()
Map.centerObject(polygon, zoom=15)
Map.setOptions('SATELLITE')
Map.addLayer(image_start_case, sar_style, f'raw start {start_date}')
Map.addLayer(image_end_case, sar_style, f'raw end {end_date}')
Map.addLayer(log_ratio, log_vis, 'changes in landscape')
Map.addLayer(change_class, log_vis, 'changes in landscape')
Map.add_layer(polygon.style(**polygon_style), {}, "mining area")
Map.add_layer(buildings.style(**buildings_style), {}, "buildings")
Map.add_legend(legend_dict=legend_dict, title="Landscape Changes", position="bottomright")
display(title)
Map

HTML(value="<h3 style='text-align:center'>Changes in landscape from 08-06-2023 to 27-06-2023 in Niedźwiedzia G…

Map(center=[50.10339412565508, 19.63649860680545], controls=(WidgetControl(options=['position', 'transparent_b…

### TIME SERIES

In [9]:
actual_date_list = []
image_list = []

rectangle = polygon.geometry().buffer(500).bounds()

for start_date, end_date in zip(filtered_date_list_start, filtered_date_list_end):
    sentinel = (
        ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
        .filterBounds(polygon.geometry())
        .filterDate(start_date, end_date)
        .sort('CLOUDY_PIXEL_PERCENTAGE')
        .first()
    )
    
    if sentinel is not None:
        qa = sentinel.select('QA60').clip(rectangle)
        cloud_mask = qa.bitwiseAnd(1 << 10).neq(0)
        
        cloud_mask = cloud_mask.reduceRegion(
            reducer = ee.Reducer.mean(),
            geometry = rectangle,
            scale = 10,
            maxPixels = 1e9
        )

        cloud_cover = cloud_mask.get('QA60')
        
        if cloud_cover.getInfo() * 100 <= 0:
            date = ee.Date(sentinel.get('system:time_start')).format('YYYY-MM-dd').getInfo()
            image_list.append(sentinel)
            actual_date_list.append(date)
        
actual_date_list

['2016-05-23',
 '2016-06-29',
 '2016-07-02',
 '2016-08-28',
 '2016-09-10',
 '2017-05-28',
 '2017-06-27',
 '2017-07-22',
 '2017-08-08',
 '2017-09-27',
 '2017-10-02',
 '2018-05-13',
 '2018-06-07',
 '2018-08-13',
 '2018-09-12',
 '2018-10-05',
 '2019-05-25',
 '2019-06-09',
 '2019-07-29',
 '2019-08-26',
 '2019-09-22',
 '2019-10-20',
 '2020-05-09',
 '2020-07-01',
 '2020-08-22',
 '2020-09-09',
 '2020-10-06',
 '2021-05-12',
 '2021-06-21',
 '2021-07-13',
 '2021-08-20',
 '2021-09-06',
 '2021-10-09',
 '2022-05-19',
 '2022-06-03',
 '2022-07-21',
 '2022-08-05',
 '2022-09-01',
 '2022-10-26',
 '2023-05-27',
 '2023-06-03',
 '2023-07-08',
 '2023-08-15',
 '2023-09-06',
 '2023-10-14',
 '2024-05-28',
 '2024-06-17',
 '2024-07-30',
 '2024-08-14',
 '2024-09-18',
 '2024-10-18']

In [10]:
bare_soil_list = []
rgb_list = []
ndvi_list = []
bare_soil_list = []
exceeded_list = []
buildings_mask_list = []

for image in image_list:
    image = image.clip(rectangle)
    rgb = image.select(['B4', 'B3', 'B2'])
    ndvi = image.normalizedDifference(['B8', 'B4'])
    bare_soil = ndvi.lt(0.18)
    rgb_list.append(rgb)
    ndvi_list.append(ndvi)
    bare_soil_list.append(bare_soil)
    

for bare_soil in bare_soil_list:
    ones = ee.Image.constant(1).clip(rectangle)
    polygon_mask = ee.Image.constant(1).clip(polygon)
    outside_mask = ones.where(polygon_mask.eq(1), 0)

    exceeded = bare_soil.updateMask(outside_mask)
    
    buildings_mask = ee.Image().byte().paint(featureCollection=buildings, color=1).clip(rectangle_case)
    buildings_mask = ones.where(buildings_mask.eq(1), 0)
    exceeded = exceeded.updateMask(buildings_mask)
    
    exceeded_list.append(exceeded)
    buildings_mask_list.append(buildings)

In [11]:
ndvi_vis = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
rgb_vis = {'min': 0, 'max': 3000}
polygon_style = {"color": "red", "fillColor": "00000000", "width": 1}
bare_soil_vis = {'min': 0, 'max': 1, 'palette': ['black', 'white']}
exceeded_vis = {'min': 0, 'max': 1, 'palette': ['black', 'yellow']}
color_palette = ['red', 'blue', 'green', 'yellow', 'purple']
buildings_style = {"color": "green", "fillColor": "00000000", "width": 1}


def show_map(index):
    m = geemap.Map()
    m.centerObject(polygon, zoom=15)
    m.add_layer(rgb_list[index], rgb_vis, "orthophotomap")
    m.add_layer(ndvi_list[index], ndvi_vis, "NDVI")
    m.add_layer(bare_soil_list[index], bare_soil_vis, "bare soil mask")
    m.add_layer(exceeded_list[index], exceeded_vis, f"exceeded {actual_date_list[index]}")
    m.add_layer(polygon.style(**polygon_style), {}, "mining area")
    m.add_layer(buildings_mask_list[index], buildings_style, "buildings")
    display(m)

slider = widgets.SelectionSlider(
    options=[(date, i) for i, date in enumerate(actual_date_list)],
    value=0,
    description='Date',
    continuous_update=False
)
widgets.interact(show_map, index=slider)


interactive(children=(SelectionSlider(continuous_update=False, description='Date', options=(('2016-05-23', 0),…

<function __main__.show_map(index)>